# Devoir 2 — Système RAG pour Textes Juridiques (Code de la Route Marocain)

**Objectif :** Concevoir et implémenter un système RAG permettant de répondre à des questions en langage naturel à partir d'une base de données de textes juridiques extraits en Devoir 1.

**Pipeline :**
1. Préparation & nettoyage des données CSV
2. Découpage en chunks exploitables
3. Génération d'embeddings + indexation FAISS
4. Intégration LLM (comparaison de 3 modèles)
5. Pipeline RAG complet
6. Évaluation (precision, recall)
7. Détection de questions hors domaine
8. Interface web interactive (Gradio)

## 1. Installation des dépendances

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers gradio scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 43.0 MB/s eta 0:00:00


## 2. Chargement & Préparation des données

In [2]:
import pandas as pd
import re
import unicodedata

# ------------------------------------------------------------------
# 2.1  Chargement du CSV produit en Devoir 2
# ------------------------------------------------------------------
df = pd.read_csv("export_final.csv")
print("Shape initiale :", df.shape)
print(df.columns.tolist())
df.head(3)

Shape initiale : (521, 16)
['article_id', 'infraction_desc', 'categorie_vehicule', 'amende_min_dh', 'amende_max_dh', 'points_retrait', 'peine_prison', 'recidive_prevue', 'suspension_permis', 'immobilisation_vehicule', 'interdiction_conduire', 'mots_cles', 'role_paragraphe_regles', 'cluster_thematique_ml', 'cluster_id', 'texte_apercu']


,article_id,infraction_desc,categorie_vehicule,amende_min_dh,amende_max_dh,points_retrait,peine_prison,recidive_prevue,suspension_permis,immobilisation_vehicule,interdiction_conduire,mots_cles,role_paragraphe_regles,cluster_thematique_ml,cluster_id,texte_apercu
0,1,لا يجوز لاي شخص ان يسوق مركبة ذات محرك او مجمو...,tous_vehicules,NaN,NaN,NaN,NaN,non,non,non,non,NaN,interdiction,circulation_routière,1,لا يجوز لأي شخص أن يسوق مركبة ذات محرك أو مجمو...
1,2,استثناء من احكام المادة الاولى اعلاه :\n1 - يج...,non_spécifié,NaN,NaN,NaN,NaN,non,non,non,non,permis,autre,permis_conduite,6,استثناء من أحكام المادة الأولى أعلاه : 1 - يج...
2,3,يجب على السايقين الحاصلين على رخصة سياقة مسلمة...,non_spécifié,NaN,NaN,NaN,NaN,non,non,non,non,permis,obligation,permis_conduite,6,يجب على السائقين الحاصلين على رخصة سياقة مسلمة...


In [3]:
# ------------------------------------------------------------------
# 2.2  Nettoyage & normalisation
# ------------------------------------------------------------------
def clean_text(text: str) -> str:
    """Normalise un texte arabe / latin extrait du CSV."""
    if pd.isna(text):
        return ""
    # Normalisation Unicode (NFC)
    text = unicodedata.normalize("NFC", str(text))
    # Suppression des caractères de contrôle et espaces multiples
    text = re.sub(r"[\x00-\x1f\x7f]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["texte_clean"] = df["infraction_desc"].apply(clean_text)

# Supprimer les lignes vides ou trop courtes
df = df[df["texte_clean"].str.len() > 20].drop_duplicates(subset="texte_clean").reset_index(drop=True)
print(f"Lignes après nettoyage : {len(df)}")
df[["article_id", "texte_clean", "role_paragraphe_regles", "categorie_vehicule"]].head(5)

Lignes après nettoyage : 477


,article_id,texte_clean,role_paragraphe_regles,categorie_vehicule
0,1,لا يجوز لاي شخص ان يسوق مركبة ذات محرك او مجمو...,interdiction,tous_vehicules
1,2,استثناء من احكام المادة الاولى اعلاه : 1 - يجو...,autre,non_spécifié
2,3,يجب على السايقين الحاصلين على رخصة سياقة مسلمة...,obligation,non_spécifié
3,4,في حالة السير الدولي ووفقا للاتفاقية الدولية ل...,autre,non_spécifié
4,5,116.14 1.16.106 بتاريخ 13 من شوال 1437 ( 18 يو...,autre,non_spécifié


In [4]:
# ------------------------------------------------------------------
# 2.3  Découpage en chunks
#       Chaque article est déjà une unité cohérente.
#       Pour les textes très longs, on applique un chunking par phrases.
# ------------------------------------------------------------------
MAX_CHUNK_CHARS = 500

def split_into_chunks(row: pd.Series, max_chars: int = MAX_CHUNK_CHARS):
    """Découpe un texte en chunks de taille max_chars avec chevauchement."""
    text = row["texte_clean"]
    article_id = row["article_id"]
    type_art   = row["role_paragraphe_regles"]

    if len(text) <= max_chars:
        return [{"chunk_id": f"{article_id}_0",
                 "article_id": article_id,
                 "role_paragraphe_regles": type_art,
                 "chunk": text}]

    # Découpage par fenêtre glissante avec chevauchement de 50 caractères
    chunks = []
    step = max_chars - 50
    for i, start in enumerate(range(0, len(text), step)):
        piece = text[start:start + max_chars]
        if len(piece) < 30:
            break
        chunks.append({
            "chunk_id":    f"{article_id}_{i}",
            "article_id":  article_id,
            "role_paragraphe_regles": type_art,
            "chunk":       piece
        })
    return chunks

all_chunks = []
for _, row in df.iterrows():
    all_chunks.extend(split_into_chunks(row))

chunks_df = pd.DataFrame(all_chunks)
print(f"Nombre total de chunks : {len(chunks_df)}")
chunks_df.head(5)

Nombre total de chunks : 477


,chunk_id,article_id,role_paragraphe_regles,chunk
0,1_0,1,interdiction,لا يجوز لاي شخص ان يسوق مركبة ذات محرك او مجمو...
1,2_0,2,autre,استثناء من احكام المادة الاولى اعلاه : 1 - يجو...
2,3_0,3,obligation,يجب على السايقين الحاصلين على رخصة سياقة مسلمة...
3,4_0,4,autre,في حالة السير الدولي ووفقا للاتفاقية الدولية ل...
4,5_0,5,autre,116.14 1.16.106 بتاريخ 13 من شوال 1437 ( 18 يو...


## 3. Indexation vectorielle (FAISS)

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# ------------------------------------------------------------------
# 3.1  Modèle d'embeddings
#       paraphrase-multilingual-MiniLM-L12-v2 supporte l'arabe et le français
# ------------------------------------------------------------------
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Modèle d'embeddings chargé : {EMBEDDING_MODEL}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modèle d'embeddings chargé : paraphrase-multilingual-MiniLM-L12-v2


In [6]:
# ------------------------------------------------------------------
# 3.2  Génération des embeddings
# ------------------------------------------------------------------
texts_to_embed = chunks_df["chunk"].tolist()
print(f"Encodage de {len(texts_to_embed)} chunks...")

embeddings = embed_model.encode(
    texts_to_embed,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"Shape des embeddings : {embeddings.shape}")

Encodage de 477 chunks...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Shape des embeddings : (477, 384)


In [7]:
# ------------------------------------------------------------------
# 3.3  Construction de l'index FAISS (IndexFlatIP = cosine similarity)
# ------------------------------------------------------------------
# Normalisation L2 pour transformer en cosine similarity
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)  # Inner Product ≃ cosine sur vecteurs normalisés
faiss_index.add(embeddings)

print(f"Index FAISS construit — {faiss_index.ntotal} vecteurs stockés")

Index FAISS construit — 477 vecteurs stockés


In [8]:
# ------------------------------------------------------------------
# 3.4  Fonction de recherche
# ------------------------------------------------------------------
def retrieve(query: str, k: int = 5) -> list[dict]:
    """Retourne les k chunks les plus pertinents pour une question."""
    q_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)
    scores, indices = faiss_index.search(q_vec, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = chunks_df.iloc[idx]
        results.append({
            "article_id":  row["article_id"],
            "role_paragraphe_regles": row["role_paragraphe_regles"],
            "chunk":       row["chunk"],
            "score":       float(score)
        })
    return results

# Test rapide
for r in retrieve("ما هي عقوبة تجاوز السرعة", k=3):
    print(f"[Article {r['article_id']} | score={r['score']:.3f}] {r['chunk'][:120]}...\n")

[Article 302 | score=0.710] يعاقب بغرامة من ماية الف...

[Article 298 | score=0.681] يعاقب بغرامة من الف ومايتين...

[Article 148 | score=0.659] يعاقب بغرامة من الفين...



## 4. Intégration LLM — Comparaison de 3 modèles

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositif utilisé : {DEVICE}")

# ------------------------------------------------------------------
# Les 3 modèles à comparer
# ------------------------------------------------------------------
LLM_CONFIGS = [
    {"name": "Qwen2.5-0.5B",   "model_id": "Qwen/Qwen2.5-0.5B-Instruct"},
    {"name": "Qwen2.5-1.5B",   "model_id": "Qwen/Qwen2.5-1.5B-Instruct"},
    {"name": "SmolLM2-1.7B",   "model_id": "HuggingFaceTB/SmolLM2-1.7B-Instruct"},
]

Dispositif utilisé : cuda


In [10]:
# ------------------------------------------------------------------
# Chargement dynamique d'un LLM — sans device_map (pas besoin d'accelerate)
# ------------------------------------------------------------------
def load_generator(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Chargement direct sans device_map => pas besoin d'accelerate
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        low_cpu_mem_usage=True
    )
    model = model.to(DEVICE)

    return pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device=0 if DEVICE == "cuda" else -1,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.3,
        pad_token_id=tokenizer.eos_token_id
    )


## 5. Pipeline RAG complet

In [11]:
# ------------------------------------------------------------------
# 5.1  Construction du prompt
# ------------------------------------------------------------------
SYSTEM_PROMPT = (
    "أنت مساعد قانوني متخصص في قانون السير على الطرق المغربي. "
    "أجب على السؤال بناءً فقط على النصوص القانونية المقدمة. "
    "إذا لم تجد الإجابة في السياق، قل ذلك بوضوح. "
    "اذكر دائماً رقم المادة القانونية المعتمدة في إجابتك."
)

def build_prompt(query: str, docs: list[dict]) -> str:
    context_parts = []
    for d in docs:
        context_parts.append(f"[المادة {d['article_id']}]\n{d['chunk']}")
    context = "\n\n".join(context_parts)

    prompt = f"""{SYSTEM_PROMPT}

السياق القانوني:
{context}

السؤال:
{query}

الجواب:"""
    return prompt


# ------------------------------------------------------------------
# 5.2  Détection de questions hors domaine
# ------------------------------------------------------------------
DOMAIN_KEYWORDS = [
    # Arabe
    "سياقة", "مركبة", "رخصة", "طريق", "سرعة", "مخالفة", "غرامة",
    "حادث", "تأمين", "نقاط", "كحول", "وقوف", "إشارة", "حزام",
    # Français (si la question est en français)
    "conduire", "permis", "vitesse", "infraction", "amende",
    "alcool", "ceinture", "feu", "stop", "route", "vehicule",
    "accident", "assurance", "stationnement"
]

def is_in_domain(query: str, threshold: float = 0.25) -> bool:
    """Retourne True si la question concerne le code de la route.

    Stratégie double :
    1. Correspondance de mots-clés du domaine
    2. Score de similarité maximal avec la base vectorielle
    """
    q_lower = query.lower()
    if any(kw in q_lower for kw in DOMAIN_KEYWORDS):
        return True

    # Vérification via le score de récupération
    top_doc = retrieve(query, k=1)
    if top_doc and top_doc[0]["score"] >= threshold:
        return True
    return False


# ------------------------------------------------------------------
# 5.3  Fonction principale RAG
# ------------------------------------------------------------------
def rag_answer(query: str, generator, k: int = 5) -> dict:
    """Pipeline RAG complet : récupération → prompt → génération → réponse."""

    # Étape 0 : vérification du domaine
    if not is_in_domain(query):
        return {
            "answer": "⚠️ هذا السؤال يبدو خارج نطاق قانون السير. لا يمكنني الإجابة عليه.",
            "sources": [],
            "out_of_domain": True
        }

    # Étape 1 : Récupération des documents pertinents
    docs = retrieve(query, k=k)

    # Étape 2 : Construction du prompt
    prompt = build_prompt(query, docs)

    # Étape 3 : Génération
    output = generator(prompt)
    full_text = output[0]["generated_text"]
    # Extraire uniquement la partie générée (après "الجواب:")
    answer = full_text.split("الجواب:")[-1].strip()

    # Étape 4 : Références
    sources = [{"article_id": d["article_id"], "score": d["score"],
                "extrait": d["chunk"][:150] + "..."} for d in docs]

    return {
        "answer": answer,
        "sources": sources,
        "out_of_domain": False
    }

In [12]:
# ------------------------------------------------------------------
# 5.4  Test avec le premier modèle (Qwen2.5-0.5B)
# ------------------------------------------------------------------
print("Chargement du modèle 1 :", LLM_CONFIGS[0]["name"])
generator_1 = load_generator(LLM_CONFIGS[0]["model_id"])

q = "ما هي عقوبة قيادة سيارة بدون رخصة؟"
result = rag_answer(q, generator_1)

print("\n🔹 السؤال :", q)
print("\n📝 الجواب :", result["answer"])
print("\n📚 المصادر :")
for src in result["sources"]:
    print(f"  - المادة {src['article_id']} (score={src['score']:.3f}) : {src['extrait']}")

Chargement du modèle 1 : Qwen2.5-0.5B


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔹 السؤال : ما هي عقوبة قيادة سيارة بدون رخصة؟

📝 الجواب : الإجابة الصحيحة هي:

1. لا ت行政处罚
2. تعاقب على استمرار في استخدام مركبة، على الطريق العمومي، خاضعة للتسجيل بمقت
3. الغاء رخصة السياقة مع المنع من اجتياز امتحان الحصول على رخصة جديدة، خلال مدة سنة الى سنتين 
4. 3 - الزامية الخضوع، على نفقتهم، لدورة في التربية على السلامة الطرقية.
5. تتعرض أيضا مرتكبو المخالفات المنصوص عليها في الفقرة الثانية من

الإجابة الصحيحة هي:

1. لا ت行政处罚
2. تعاقب على استمرار في استخدام مركبة، على الطريق العمومي، خاضعة للتسجيل بمقت
3. الغاء رخصة السياقة مع المنع من اجتياز امتحان الحصول على رخصة جديدة، خلال مدة سنة الى سنتين 
4. 3 - الزامية الخضوع، على نفقتهم، لدورة في التربية على السلامة الطرقية.
5. تتعرض أيضا مرتكبو المخالف

📚 المصادر :
  - المادة 185 (score=0.776) : او في حالة ارتكاب مخالفة للاحكام المقررة تطبيقا للمواد 46 و 47 و 48 من هذا القانون. اذا ادى المخالف مبلغ الغرامة التصالحية والجزافية بصفة نهايية داخل ...
  - المادة 166 (score=0.738) : - 1 اعلاه، لتوقيف رخصة السياقة لمدة سنة الى سنتين. ولا ترج

## 6. Comparaison des 3 LLMs

In [ ]:
import time

# Questions de test
TEST_QUERIES = [
    "ما هي شروط الحصول على رخصة السياقة؟",
    "ما هي عقوبة تجاوز السرعة المسموح بها؟",
    "كيف يتم خصم النقاط من رخصة السياقة؟",
]

comparison_results = []

for cfg in LLM_CONFIGS:
    print(f"\n{'='*60}")
    print(f"🤖 Modèle : {cfg['name']}")
    print(f"{'='*60}")

    try:
        gen = load_generator(cfg["model_id"])

        for q in TEST_QUERIES:
            t0 = time.time()
            res = rag_answer(q, gen)
            elapsed = time.time() - t0

            entry = {
                "model": cfg["name"],
                "query": q,
                "answer": res["answer"],
                "latency_s": round(elapsed, 2),
                "n_sources": len(res["sources"])
            }
            comparison_results.append(entry)

            print(f"\n❓ {q}")
            print(f"⏱️  Latence : {elapsed:.2f}s")
            print(f"✅ Réponse : {res['answer'][:300]}...")

        # Libérer la mémoire GPU
        del gen
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    except Exception as e:
        print(f"❌ Erreur pour {cfg['name']}: {e}")

comparison_df = pd.DataFrame(comparison_results)
print("\n\n📊 Tableau comparatif des latences :")
print(comparison_df.groupby("model")["latency_s"].agg(["mean", "min", "max"]))


🤖 Modèle : Qwen2.5-0.5B


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



❓ ما هي شروط الحصول على رخصة السياقة؟
⏱️  Latence : 9.35s
✅ Réponse : 1. يجب أن تتضمن رخصة السياقة رخصة حملها.
2. يجب أن يكون الحامل المحرر فيها رخصة حملها.
3. يجب أن يكون الحامل المحرر فيها رخصة حملها.
4. يجب أن يكون الحامل المحرر فيها رخصة حملها.
5. يجب أن تكون رخصة حملها موجودة على محضر مخاطر.
6. يجب أن يكون الحامل المحرر فيها رخصة حملها.
7. يجب أن يكون الحامل المح...


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

## 7. Évaluation des performances (Precision & Recall)

In [ ]:
from sklearn.metrics import precision_score, recall_score

# ------------------------------------------------------------------
# 7.1  Jeu de test annoté manuellement
#       Format : {"query": ..., "relevant_articles": [id1, id2, ...]}
# ------------------------------------------------------------------
EVAL_SET = [
    {"query": "ما هي شروط رخصة السياقة",
     "relevant_articles": [1, 7, 10, 11, 23]},
    {"query": "عقوبة القيادة بدون رخصة",
     "relevant_articles": [1, 6, 9]},
    {"query": "خصم نقاط رخصة السياقة",
     "relevant_articles": [22, 24, 27, 28]},
    {"query": "الفحص الطبي للحصول على رخصة",
     "relevant_articles": [12, 13, 14, 15, 16]},
    {"query": "رخصة السياقة الدولية",
     "relevant_articles": [4]},
]

ALL_ARTICLE_IDS = sorted(chunks_df["article_id"].unique().tolist())


def evaluate_retriever(eval_set: list, k: int = 5) -> dict:
    """Calcule Precision@k et Recall@k pour le retriever."""
    precisions, recalls = [], []

    for item in eval_set:
        retrieved_docs  = retrieve(item["query"], k=k)
        retrieved_ids   = set(d["article_id"] for d in retrieved_docs)
        relevant_ids    = set(item["relevant_articles"])

        tp = len(retrieved_ids & relevant_ids)
        precision = tp / k if k > 0 else 0
        recall    = tp / len(relevant_ids) if relevant_ids else 0

        precisions.append(precision)
        recalls.append(recall)

        print(f"\nQ: {item['query']}")
        print(f"  Récupérés  : {sorted(retrieved_ids)}")
        print(f"  Pertinents : {sorted(relevant_ids)}")
        print(f"  Precision@{k}={precision:.2f}  Recall@{k}={recall:.2f}")

    return {
        f"Precision@{k}": round(sum(precisions) / len(precisions), 3),
        f"Recall@{k}":    round(sum(recalls)    / len(recalls),    3),
    }


metrics_k5 = evaluate_retriever(EVAL_SET, k=5)
metrics_k10 = evaluate_retriever(EVAL_SET, k=10)

print("\n📊 Métriques du Retriever :")
print("  k=5  :", metrics_k5)
print("  k=10 :", metrics_k10)

## 8. Détection de questions hors domaine

In [ ]:
# ------------------------------------------------------------------
# Tests de détection hors domaine
# ------------------------------------------------------------------
test_questions = [
    # Dans le domaine ✅
    ("ما هي عقوبة تجاوز السرعة؟",            True),
    ("كيف أحصل على رخصة السياقة؟",           True),
    ("Quel est l'amende pour excès de vitesse ?", True),
    # Hors domaine ❌
    ("ما هو الناتج المحلي الإجمالي للمغرب؟",  False),
    ("Comment préparer un couscous ?",        False),
    ("What is the capital of France?",         False),
]

print("Tests de détection hors domaine :\n")
correct = 0
for q, expected in test_questions:
    predicted = is_in_domain(q)
    ok = predicted == expected
    correct += int(ok)
    emoji = "✅" if ok else "❌"
    print(f"{emoji} [{('IN' if predicted else 'OUT')}] {q}")

print(f"\nPrécision de détection : {correct}/{len(test_questions)}")

## 9. Interface Web interactive (Gradio)

In [ ]:
# ------------------------------------------------------------------
# Charger le modèle par défaut pour l'interface
# (modifiez selon les ressources disponibles)
# ------------------------------------------------------------------
DEFAULT_MODEL_ID = LLM_CONFIGS[0]["model_id"]
print(f"Chargement du modèle par défaut : {DEFAULT_MODEL_ID}")
default_generator = load_generator(DEFAULT_MODEL_ID)

In [ ]:
import gradio as gr

# ------------------------------------------------------------------
# Fonction appelée par Gradio
# ------------------------------------------------------------------
def gradio_rag(question: str, top_k: int = 5):
    if not question.strip():
        return "⚠️ الرجاء إدخال سؤال.", ""

    result = rag_answer(question, default_generator, k=top_k)

    answer_text = result["answer"]

    if result["out_of_domain"]:
        sources_text = "لا توجد مصادر — السؤال خارج النطاق."
    else:
        sources_lines = []
        for s in result["sources"]:
            sources_lines.append(
                f"**المادة {s['article_id']}** (تطابق: {s['score']:.3f})\n{s['extrait']}\n"
            )
        sources_text = "\n---\n".join(sources_lines)

    return answer_text, sources_text


# ------------------------------------------------------------------
# Interface Gradio
# ------------------------------------------------------------------
with gr.Blocks(title="مساعد قانون السير المغربي", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🚗 مساعد قانون السير على الطرق المغربي
        **Système RAG — Code de la Route Marocain (Loi 52-05)**
        Posez votre question en arabe ou en français.
        """
    )

    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(
                label="❓ السؤال / Question",
                placeholder="ما هي عقوبة القيادة بدون رخصة ؟",
                lines=3
            )
            top_k_slider = gr.Slider(
                minimum=1, maximum=10, value=5, step=1,
                label="عدد المستندات المسترجعة (k)"
            )
            submit_btn = gr.Button("🔍 البحث والإجابة", variant="primary")

        with gr.Column(scale=3):
            answer_output = gr.Textbox(label="📝 الجواب / Réponse", lines=8)
            sources_output = gr.Markdown(label="📚 المصادر القانونية / Sources")

    gr.Examples(
        examples=[
            ["ما هي شروط الحصول على رخصة السياقة؟", 5],
            ["ما هي عقوبة تجاوز السرعة؟", 5],
            ["كيف يتم خصم النقاط من رخصة السياقة؟", 5],
            ["Quelle est la sanction pour conduite sans permis ?", 5],
            ["ما هو الطقس اليوم؟", 5],  # question hors domaine
        ],
        inputs=[question_input, top_k_slider]
    )

    submit_btn.click(
        fn=gradio_rag,
        inputs=[question_input, top_k_slider],
        outputs=[answer_output, sources_output]
    )

demo.launch(share=True)  # share=True génère un lien public Gradio

## 10. Résumé du système

| Composant | Choix | Justification |
|-----------|-------|---------------|
| **Embeddings** | `paraphrase-multilingual-MiniLM-L12-v2` | Support arabe + français, léger |
| **Vector Store** | FAISS (IndexFlatIP) | Cosine similarity, rapide en CPU |
| **Chunking** | 500 chars, overlap 50 | Articles courts → 1 chunk, longs → fenêtre glissante |
| **LLM 1** | Qwen2.5-0.5B-Instruct | Modèle léger, baseline |
| **LLM 2** | Qwen2.5-1.5B-Instruct | Meilleure compréhension, plus lent |
| **LLM 3** | SmolLM2-1.7B-Instruct | Modèle alternatif pour comparaison |
| **Out-of-domain** | Mots-clés + score FAISS | Rejet si score < seuil |
| **Interface** | Gradio | Interface web simple, déploiement en 1 ligne |

### Points forts du système
- Supporte les questions en **arabe et en français**
- Cite toujours le **numéro de la mادة** (article) source
- Détecte et rejette les **questions hors domaine**
- Évaluation quantitative via **Precision@k** et **Recall@k**